# Exploration — Stratégies d'assignation du label genre

**Contexte :** La majorité des films ont plusieurs genres. Pour entraîner un classifieur mono-label, il faut choisir **comment assigner un seul genre** à chaque film.

**Référence issue de `single_genre_analysis.ipynb` :** Macro F1 **0.67**.

## Stratégies testées

| # | Stratégie | Description |
|---|---|---|
| 0 | **Baseline** | Premier genre listé (stratégie actuelle) |
| 2 | **Priorité fixe** | Si Animation dans la liste → Animation, sinon Horror, sinon Drama |
| 3 | **Genre le plus rare** | Parmi les genres listés, on prend le moins fréquent dans le dataset |
| 4 | **Multi-instance** | Les films multi-genres sont dupliqués : une ligne par genre correspondant à nos 3 genres |

> **Critère de décision :** Macro F1 > 0.67 → on adopte la stratégie. Sinon → on garde S0 (premier genre).

## Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report

RANDOM_STATE = 42
SELECTED_GENRES = ['Animation', 'Horror', 'Drama']
continuous_features  = ['rating', 'total_votes', 'popularity']
passthrough_features = ['is_english', 'cast_count', 'release_month', 'release_year']
features = continuous_features + passthrough_features

In [2]:
df = pd.read_csv("hf://datasets/HenryWaltson/TMDB-IMDB-Movies-Dataset/TMDB  IMDB Movies Dataset.csv")
df = df.drop_duplicates()
df = df.drop(columns=['backdrop_path', 'keywords', 'homepage', 'tconst', 'overview', 'poster_path', 'tagline'])
df = df[df['release_date'].notna()]

total_votes = df['vote_count'] + df['numVotes']
df['rating']      = (df['vote_average'] * df['vote_count'] + df['averageRating'] * df['numVotes']) / total_votes
df['total_votes'] = total_votes
df = df.drop(columns=['vote_count', 'numVotes', 'vote_average', 'averageRating'])

df['is_english']    = (df['original_language'] == 'en').astype(int)
df['cast_count']    = df['cast'].fillna('').apply(lambda x: len(x.split(',')) if x else 0)
df['release_month'] = pd.to_datetime(df['release_date']).dt.month
df['release_year']  = pd.to_datetime(df['release_date']).dt.year

df_with_genre = df[df['genres'].notna()].copy()

# Fréquence globale des genres (utile pour stratégie 3)
genre_global_freq = (
    df_with_genre['genres'].str.split(', ').explode()
    .value_counts(normalize=True)
    .to_dict()
)

print(f"Dataset prêt : {len(df_with_genre):,} films avec genre(s)")

/Users/adam/Desktop/ECE/ING4/S8/Apprentissage et Estimation Bayesienne/Projet/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset prêt : 351,298 films avec genre(s)


In [3]:
def make_pipeline():
    preprocessor = ColumnTransformer([
        ('scale', RobustScaler(), continuous_features),
        ('pass',  'passthrough',  passthrough_features)
    ])
    return Pipeline([('preprocessor', preprocessor), ('model', GaussianNB())])

def evaluate(df_labeled, label, verbose=True):
    """Undersample, split, entraîne et évalue. Retourne le rapport."""
    df_sel = df_labeled[df_labeled['genre'].isin(SELECTED_GENRES)].copy()
    cap = df_sel['genre'].value_counts().min()
    df_bal = df_sel.groupby('genre', group_keys=False).sample(n=cap, random_state=RANDOM_STATE)

    X_tr, X_te, y_tr, y_te = train_test_split(
        df_bal[features], df_bal['genre'],
        test_size=0.2, stratify=df_bal['genre'], random_state=RANDOM_STATE
    )
    pipe = make_pipeline()
    pipe.fit(X_tr, y_tr)
    report = classification_report(y_te, pipe.predict(X_te), output_dict=True)

    if verbose:
        print(f"  Films utilisés : {len(df_bal):,} ({cap:,}/genre)")
        print(classification_report(y_te, pipe.predict(X_te)))

    return {
        'label':    label,
        'n':        len(df_bal),
        'macro_f1': report['macro avg']['f1-score'],
        'anim_f1':  report['Animation']['f1-score'],
        'drama_f1': report['Drama']['f1-score'],
        'horror_f1':report['Horror']['f1-score'],
        'acc':      report['accuracy'],
    }

results = []

---
## Stratégie 0 — Baseline : premier genre listé

In [4]:
df_s0 = df_with_genre.copy()
df_s0['genre'] = df_s0['genres'].str.split(',').str[0].str.strip()

print("=== Stratégie 0 — Premier genre ===")
results.append(evaluate(df_s0, 'S0 — Premier genre'))

=== Stratégie 0 — Premier genre ===
  Films utilisés : 58,203 (19,401/genre)
              precision    recall  f1-score   support

   Animation       0.73      0.73      0.73      3880
       Drama       0.60      0.72      0.66      3880
      Horror       0.71      0.57      0.63      3881

    accuracy                           0.67     11641
   macro avg       0.68      0.67      0.67     11641
weighted avg       0.68      0.67      0.67     11641



---
## Stratégie 2 — Priorité fixe

On définit une hiérarchie basée sur les profils numériques les plus distincts :
**Animation > Horror > Drama**

Si un film a Animation dans sa liste → Animation. Sinon Horror. Sinon Drama. Sinon ignoré.

In [6]:
PRIORITY = ['Animation', 'Horror', 'Drama']

def assign_priority(genres_str):
    genres = [g.strip() for g in genres_str.split(',')]
    for g in PRIORITY:
        if g in genres:
            return g
    return None

df_s2 = df_with_genre.copy()
df_s2['genre'] = df_s2['genres'].apply(assign_priority)
df_s2 = df_s2[df_s2['genre'].notna()]

print("=== Stratégie 2 — Priorité fixe (Animation > Horror > Drama) ===")
results.append(evaluate(df_s2, 'S2 — Priorité fixe'))

=== Stratégie 2 — Priorité fixe (Animation > Horror > Drama) ===
  Films utilisés : 75,753 (25,251/genre)
              precision    recall  f1-score   support

   Animation       0.72      0.67      0.69      5050
       Drama       0.55      0.80      0.65      5050
      Horror       0.73      0.45      0.56      5051

    accuracy                           0.64     15151
   macro avg       0.67      0.64      0.63     15151
weighted avg       0.67      0.64      0.63     15151



---
## Stratégie 3 — Genre le plus rare

Parmi les genres listés pour un film, on prend celui qui est le **moins fréquent globalement** dans le dataset.

Intuition : Drama est souvent listé "par défaut" ou en complément — le genre rare est potentiellement le genre principal.

In [7]:
def assign_rarest(genres_str):
    genres = [g.strip() for g in genres_str.split(',')]
    # Garder uniquement nos 3 genres cibles
    genres_target = [g for g in genres if g in SELECTED_GENRES]
    if not genres_target:
        return None
    # Prendre le moins fréquent globalement
    return min(genres_target, key=lambda g: genre_global_freq.get(g, 1))

df_s3 = df_with_genre.copy()
df_s3['genre'] = df_s3['genres'].apply(assign_rarest)
df_s3 = df_s3[df_s3['genre'].notna()]

print("Distribution avec stratégie genre le plus rare :")
print(df_s3['genre'].value_counts())
print()
print("=== Stratégie 3 — Genre le plus rare ===")
results.append(evaluate(df_s3, 'S3 — Genre le plus rare'))

Distribution avec stratégie genre le plus rare :
genre
Drama        132472
Horror        28475
Animation     25251
Name: count, dtype: int64

=== Stratégie 3 — Genre le plus rare ===
  Films utilisés : 75,753 (25,251/genre)
              precision    recall  f1-score   support

   Animation       0.72      0.67      0.69      5050
       Drama       0.55      0.80      0.65      5050
      Horror       0.73      0.45      0.56      5051

    accuracy                           0.64     15151
   macro avg       0.67      0.64      0.63     15151
weighted avg       0.67      0.64      0.63     15151



---
## Stratégie 4 — Multi-instance

Pour les films multi-genres, on crée **une ligne par genre cible présent** dans la liste.

Exemple : *Interstellar* (`Adventure, Drama, Science Fiction`) → une ligne Drama.
Un film `Animation, Drama` → deux lignes : une Animation, une Drama.

Intuition : on ne perd aucune information, mais on accepte que certains films apparaissent plusieurs fois avec des labels différents.

In [8]:
rows = []
for _, row in df_with_genre.iterrows():
    genres = [g.strip() for g in row['genres'].split(',')]
    for g in genres:
        if g in SELECTED_GENRES:
            new_row = row.copy()
            new_row['genre'] = g
            rows.append(new_row)

df_s4 = pd.DataFrame(rows)

print("Distribution avec stratégie multi-instance :")
print(df_s4['genre'].value_counts())
print()
print("=== Stratégie 4 — Multi-instance ===")
results.append(evaluate(df_s4, 'S4 — Multi-instance'))

Distribution avec stratégie multi-instance :
genre
Drama        138078
Horror        29513
Animation     25257
Name: count, dtype: int64

=== Stratégie 4 — Multi-instance ===
  Films utilisés : 75,771 (25,257/genre)
              precision    recall  f1-score   support

   Animation       0.69      0.69      0.69      5051
       Drama       0.55      0.76      0.64      5052
      Horror       0.72      0.45      0.55      5052

    accuracy                           0.63     15155
   macro avg       0.65      0.63      0.63     15155
weighted avg       0.65      0.63      0.63     15155



---
## Récapitulatif

In [9]:
df_results = pd.DataFrame(results)

print(f"{'Stratégie':<35} {'Anim':>5} {'Drama':>5} {'Horr':>5} {'Macro':>5} {'Acc':>7} {'N films':>10}")
print('-' * 80)
for _, r in df_results.sort_values('macro_f1', ascending=False).iterrows():
    print(f"{r['label']:<35} {r['anim_f1']:.2f}  {r['drama_f1']:.2f}  {r['horror_f1']:.2f}  {r['macro_f1']:.2f}  {r['acc']:.2%} {int(r['n']):>10,}")

Stratégie                            Anim Drama  Horr Macro     Acc    N films
--------------------------------------------------------------------------------
S0 — Premier genre                  0.73  0.66  0.63  0.67  67.42%     58,203
S1 — Genre unique seulement         0.73  0.65  0.62  0.67  67.51%     31,059
S2 — Priorité fixe                  0.69  0.65  0.56  0.63  63.99%     75,753
S3 — Genre le plus rare             0.69  0.65  0.56  0.63  63.99%     75,753
S4 — Multi-instance                 0.69  0.64  0.55  0.63  63.40%     75,771


---
## Conclusions

### Récapitulatif des résultats

| Stratégie | Anim F1 | Drama F1 | Horror F1 | Macro F1 | Acc | N films |
|---|---|---|---|---|---|---|
| **S0 — Premier genre** | **0.73** | **0.66** | **0.63** | **0.67** | **67.4%** | 58k |
| S2 — Priorité fixe | 0.69 | 0.65 | 0.56 | 0.63 | 64.0% | 75k |
| S3 — Genre le plus rare | 0.69 | 0.65 | 0.56 | 0.63 | 64.0% | 75k |
| S4 — Multi-instance | 0.69 | 0.64 | 0.55 | 0.63 | 63.4% | 75k |

### Analyse

**Aucune stratégie alternative ne bat le baseline (S0).** Toutes les stratégies S2/S3/S4 dégradent le Macro F1 de **0.67 → 0.63**, soit -4 points.

Points notables :
- **S2 et S3 donnent exactement les mêmes résultats** — cela signifie que pour nos 3 genres, le genre "le plus rare" est aussi celui qui serait prioritaire selon notre hiérarchie fixe. Animation et Horror sont naturellement rares vs Drama.
- **S2/S3/S4 augmentent le dataset** (75k vs 58k) mais **dégradent Horror** (F1 0.63 → 0.55) — plus de données ne compense pas le bruit de label introduit.
- Le problème vient de Drama : en permettant à un film d'être labellisé Drama parce qu'il a Drama dans ses genres secondaires, on dilue le signal numérique propre à Drama.

### Décision finale

**On garde S0 — premier genre listé.** C'est la stratégie la plus simple et la plus performante.

La question de l'ordre des genres dans le dataset reste ouverte, mais les résultats montrent que cet ordre est **corrélé avec le profil numérique** du film — le premier genre listé est donc un proxy raisonnable du genre principal.

> **À propager dans `main.ipynb`** : aucun changement nécessaire sur la stratégie de label — S0 est déjà la stratégie actuelle.